<a href="https://colab.research.google.com/github/lautarodibartolo-ae/e8102-exploracion-de-datos/blob/main/bloque-2-distribuciones-y-analisis-de-datos/ejercitacion_bloque_2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Abrir en Colab"/></a>


# Bloque 2 — Ejercitación

**Asignatura E8102 · Exploración de datos**

Esta ejercitación repasa la unidad 4, Análisis de datos, sobre la **Encuesta Permanente de
Hogares** del INDEC del segundo trimestre de 2025: 46.086 personas y 16.169 hogares de 32
aglomerados urbanos.

**No hace falta saber programar.** Cada ejercicio empieza con un ejemplo resuelto sobre otra
variable, con la explicación de cada instrucción nueva, y después pide hacer lo mismo con una
variable nueva, cambiando pocas palabras. Son trece ejercicios en el orden del apunte, más un
ejercicio 0 de práctica, pensados para unas tres horas.

Los ejercicios 6, 8 y 12 son **opcionales**: si no te alcanza el tiempo, salteálos enteros, con su
ejemplo, y el resto del notebook funciona igual.

No se entrega y no se corrige. La consigna en PDF, `E8102_Ejercitacion_Bloque_2.pdf`, tiene los
mismos ejercicios, una guía corta de Python y de gráficos, y al final las soluciones.


## Cómo se usa

Antes de tocar nada: `Archivo` → `Guardar una copia en Drive`. Si no, los cambios se pierden.

Cada ejercicio tiene cuatro partes:

1. **Cómo se hace.** Un ejemplo resuelto, con la explicación de cada instrucción nueva. La celda
   de código ya está completa: solo hay que ejecutarla con `Shift + Enter` y mirar el resultado.
2. **Ahora vos.** Una celda casi igual al ejemplo, con huecos marcados `COMPLETAR`. Reemplazá cada
   `COMPLETAR` por lo que corresponda, casi siempre copiando del ejemplo, y ejecutá la celda. La
   sección 0 explica cómo se llena un hueco.
3. **Control.** Una celda que dice si cada resultado es correcto. No muestra la respuesta: marca ✔
   o ✘ y dice qué revisar. Ejecutalo cuando la celda del ejercicio corra sin un error en rojo.
4. **Pregunta.** En la mayoría de los ejercicios, una pregunta para contestar con tus palabras. Para
   escribir, hacé doble clic sobre *Tu respuesta:*. No tiene control, porque la respuesta es una
   interpretación.

Los ejercicios usan resultados de los anteriores, así que conviene hacerlos en orden y sin
saltear ningún ejemplo. Los opcionales son la excepción: ninguna celda posterior usa lo que crean.

Si cerrás el notebook, o si Colab avisa que el entorno se desconectó después de un rato sin uso,
las variables se borran. Ejecutá todo de nuevo con `Ctrl + F9`, o desde el menú `Entorno de
ejecución`. La ejecución se frena en la primera celda con un error.


## Preparación

Ejecutá estas tres celdas una vez, en orden, con el botón ▶ que aparece a la izquierda de cada
título. No hace falta leerlas, y por eso su código está oculto. La primera carga las herramientas,
la segunda descarga los datos de la EPH, y la tercera prepara los controles. La primera vez, Colab
puede avisar que el notebook no lo creó Google: elegí `Ejecutar de todos modos`.


In [ ]:
#@title Herramientas { run: "auto", display-mode: "form" }

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.transforms as mtransforms
from matplotlib.patches import Circle, Polygon, Rectangle
from cycler import cycler
from fractions import Fraction
from io import StringIO
import scipy
from scipy import stats

# La paleta de las ilustraciones del apunte, para que los gráficos se vean como el mismo material.
CREMA, TEAL, TEAL_PALIDO, MOSTAZA, CORAL, TINTA = (
    "#FEF8EF", "#61A9A1", "#D0E5DE", "#EEA92F", "#F07C52", "#32435D")

plt.rcParams.update({
    "figure.facecolor": CREMA, "axes.facecolor": CREMA, "savefig.facecolor": CREMA,
    "axes.edgecolor": TINTA, "axes.labelcolor": TINTA, "text.color": TINTA,
    "xtick.color": TINTA, "ytick.color": TINTA,
    "axes.spines.top": False, "axes.spines.right": False,
    "axes.prop_cycle": cycler(color=[TEAL, CORAL, MOSTAZA, TINTA]),
    "axes.grid": True, "grid.color": TEAL_PALIDO, "grid.linewidth": 0.8, "axes.axisbelow": True,
    "figure.figsize": (9, 4.5), "figure.dpi": 100,
    "axes.titlesize": 13, "axes.titleweight": "bold", "axes.titlelocation": "left",
    "legend.frameon": False, "font.size": 11,
})


def num(x, decimales=4):
    """Número con coma decimal, como en el apunte."""
    if x is None or (isinstance(x, float) and np.isnan(x)):
        return "—"
    if abs(x) < 0.5 * 10 ** -decimales:
        x = 0.0   # evita el "-0,0000"
    return f"{x:,.{decimales}f}".replace(",", "X").replace(".", ",").replace("X", ".")


def frac(k, n):
    """'k/n = 0,xxxx' con la fracción sin simplificar, que es como la escribe el apunte."""
    return f"{k}/{n} = {num(k / n)}"


print("numpy", np.__version__, "| pandas", pd.__version__, "| scipy", scipy.__version__, "| todo listo")


from pathlib import Path


def en_posicion(valores, posicion):
    """El valor que ocupa una posición de una lista ordenada, contando desde 1 como el apunte."""
    if posicion != int(posicion) or not 1 <= posicion <= len(valores):
        raise ValueError(f"La posición tiene que ser un número entero entre 1 y {len(valores)}, y es {posicion}.")
    return valores[int(posicion) - 1]


def por_grupo(tabla, columna, grupo, orden):
    """Los valores de una columna separados por categoría, en el orden dado: lo que pide el boxplot."""
    return [tabla.loc[tabla[grupo] == categoria, columna] for categoria in orden]


def caja(ax, grupos, etiquetas):
    """Un boxplot horizontal con la paleta del apunte."""
    estilo = dict(patch_artist=True, widths=0.55,
                  boxprops=dict(facecolor=TEAL_PALIDO, edgecolor=TINTA),
                  medianprops=dict(color=CORAL, linewidth=2.2),
                  whiskerprops=dict(color=TINTA), capprops=dict(color=TINTA),
                  flierprops=dict(marker="o", markerfacecolor=CORAL, markeredgecolor=CORAL,
                                  markersize=4, alpha=0.5))
    # `vert` pasó a llamarse `orientation` en matplotlib 3.10; `tick_labels` exige 3.9 o más.
    version = tuple(int(parte) for parte in plt.matplotlib.__version__.split(".")[:2])
    estilo["orientation" if version >= (3, 10) else "vert"] = "horizontal" if version >= (3, 10) else False
    return ax.boxplot(grupos, tick_labels=etiquetas, **estilo)


In [ ]:
#@title Los datos de la EPH { run: "auto", display-mode: "form" }

URL_DATOS = "https://raw.githubusercontent.com/lautarodibartolo-ae/e8102-exploracion-de-datos/main/bloque-2-distribuciones-y-analisis-de-datos/datos/"


def leer(nombre):
    """Lee el archivo de la carpeta datos/ si existe, y si no del repositorio publicado."""
    local = Path("datos") / nombre
    if local.exists():
        return pd.read_csv(local)
    try:
        return pd.read_csv(URL_DATOS + nombre)
    except OSError as e:
        raise RuntimeError(f"No se pudo descargar {nombre} ({e}). Revisá la conexión a internet "
                           "y volvé a ejecutar esta celda.") from None


personas = leer("eph_2t2025_personas.csv")
hogares = leer("eph_2t2025_hogares.csv")
trimestres = leer("eph_trimestres.csv")

print(f"{num(len(personas), 0)} personas, {num(len(hogares), 0)} hogares, {len(trimestres)} trimestres")


In [ ]:
#@title Controles de los ejercicios { run: "auto", display-mode: "form" }

# Las respuestas se calculan acá, a partir de los datos, y no se muestran.
_OC = personas[personas["condicion_actividad"] == "ocupado"]
_ING = _OC["ingreso_ocupacion_principal"]
_VAL = _ING[_ING != -9]
_HOR = _OC["horas_semanales"]
_HV = _HOR[_HOR != 999]
_MIE = hogares["miembros"].value_counts().sort_index()
_ORDEN = ["sin instrucción", "primario incompleto", "primario completo", "secundario incompleto",
          "secundario completo", "superior o universitario incompleto",
          "superior o universitario completo"]


def _cerca(a, b, tolerancia=1e-6):
    try:
        return abs(float(a) - float(b)) <= tolerancia * max(1.0, abs(float(b)))
    except (TypeError, ValueError):
        return False


def _hueco(valor):
    return isinstance(valor, str) and "COMPLETAR" in valor


def _numero(valor):
    return isinstance(valor, (int, float, np.number)) and not isinstance(valor, bool)


def _escala(valor):
    texto = str(valor).strip().lower()
    for con, sin in (("á", "a"), ("é", "e"), ("í", "i"), ("ó", "o"), ("ú", "u")):
        texto = texto.replace(con, sin)
    return texto.replace("de ", "").replace("escala ", "")


def _pruebas_0(g):
    mujeres = personas[personas["sexo"] == "mujer"]
    yield "`cantidad_de_mujeres`", g["cantidad_de_mujeres"] == len(mujeres)
    yield "`edad_media_mujeres`", _cerca(g["edad_media_mujeres"], mujeres["edad"].mean())


def _pruebas_1(g):
    esperado = {"region": "nominal", "sexo": "nominal", "nivel_educativo": "ordinal",
                "edad": "razon", "horas_semanales": "razon", "ingreso_ocupacion_principal": "razon"}
    e = g["escala"]
    for columna, escala in esperado.items():
        dado = e.get(columna, "COMPLETAR")
        yield (f"`{columna}`" + (": falta completar" if _hueco(dado) else ""),
               not _hueco(dado) and _escala(dado) == escala)


def _pruebas_2(g):
    yield "`ingreso` es la columna del ingreso de los ocupados", len(g["ingreso"]) == len(_ING) and (
        g["ingreso"].fillna(-1) == _ING.fillna(-1)).all()
    yield "`cantidad_centinelas`: cuántos -9 hay", g["cantidad_centinelas"] == (_ING == -9).sum()
    yield "`cantidad_ceros`: cuántos ceros hay", g["cantidad_ceros"] == (_ING == 0).sum()
    yield "`ingreso_valido` tiene un valor por ocupado que declaró su ingreso", len(g["ingreso_valido"]) == len(_VAL)
    yield "`ingreso_valido` conserva los ceros", (g["ingreso_valido"] == 0).sum() == (_VAL == 0).sum()
    yield "`media_correcta`", _cerca(g["media_correcta"], _VAL.mean())


def _pruebas_3(g):
    yield "`fi` cuenta los miembros de los hogares", int(g["fi"].sum()) == len(hogares)
    yield "`fi` está ordenada por cantidad de miembros", list(g["fi"].index) == list(_MIE.index)
    yield "`fri` suma 1", _cerca(g["fri"].sum(), 1)
    yield "`Fi` termina en el total de hogares", g["Fi"].iloc[-1] == len(hogares)
    yield "`mediana_leida`", _cerca(g["mediana_leida"], hogares["miembros"].median())
    hasta_4 = _MIE.loc[:4].sum()
    sin_miles = _cerca(g["hogares_de_hasta_4"], hasta_4 / 1000)
    yield ("`hogares_de_hasta_4`" + (": escribí el número sin el punto de los miles" if sin_miles else ""),
           _cerca(g["hogares_de_hasta_4"], hasta_4))


def _pruebas_4(g):
    yield "`horas` no tiene el 999", len(g["horas"]) == len(_HV) and (g["horas"] != 999).all()
    yield "`k_sturges` usa la cantidad de datos", _cerca(g["k_sturges"], 1 + 3.322 * np.log10(len(_HV)))
    yield "`clases_horas`: ningún dato quedó afuera de los intervalos", g["clases_horas"].isna().sum() == 0
    yield "los intervalos son cerrados a la izquierda", bool(g["clases_horas"].cat.categories.closed == "left")
    yield "`ultimo_borde`: es el menor borde que deja adentro al 140", len(g["fi_horas"]) == 15
    marcas = np.arange(5, 150, 10)
    frec = pd.cut(_HV, bins=range(0, 151, 10), right=False).value_counts().sort_index().to_numpy()
    yield "`media_agrupada_horas`", _cerca(g["media_agrupada_horas"], (marcas * frec).sum() / len(_HV))


def _pruebas_5(g):
    yield "`media`", _cerca(g["media"], _VAL.mean())
    yield "`mediana`", _cerca(g["mediana"], _VAL.median())
    yield "`modo`", _cerca(g["modo"], _VAL.mode().iloc[0])
    yield "`proporcion_bajo_la_media`", _cerca(g["proporcion_bajo_la_media"], (_VAL < _VAL.mean()).mean())
    yield "`ingreso_tipico`: la medida que se informa", _cerca(g["ingreso_tipico"], _VAL.median())


def _pruebas_6(g):
    once = np.sort(_OC.loc[(_OC["region"] == "Patagonia") & (_ING != -9),
                           "ingreso_ocupacion_principal"].head(11).to_numpy())
    yield "`q1_formula`", _cerca(g["q1_formula"], once[2])
    yield "`q2_formula`", _cerca(g["q2_formula"], once[5])
    yield "`q3_formula`", _cerca(g["q3_formula"], once[8])


def _pruebas_7(g):
    q1, q3 = _VAL.quantile([0.25, 0.75])
    yield "`rango`", _cerca(g["rango"], _VAL.max() - _VAL.min())
    yield "`desvio_n`, con divisor n", _cerca(g["desvio_n"], _VAL.std(ddof=0))
    yield "`q1` y `q3`", _cerca(g["q1"], q1) and _cerca(g["q3"], q3)
    yield "`ric`", _cerca(g["ric"], q3 - q1)
    yield "`cv_ingreso`", _cerca(g["cv_ingreso"], _VAL.std(ddof=0) / _VAL.mean())
    yield "`dispersion_tipica`: la medida que se informa", _cerca(g["dispersion_tipica"], q3 - q1)


def _pruebas_8(g):
    s = _VAL.std(ddof=0)
    yield "`a3`", _cerca(g["a3"], stats.skew(_VAL))
    yield "`a4`: la kurtosis α₄, no el exceso", _cerca(g["a4"], stats.kurtosis(_VAL, fisher=False))
    yield "`ap1`", _cerca(g["ap1"], (_VAL.mean() - _VAL.mode().iloc[0]) / s)
    yield "`ap2`", _cerca(g["ap2"], 3 * (_VAL.mean() - _VAL.median()) / s)


def _pruebas_9(g):
    z = (_VAL - _VAL.mean()) / _VAL.std(ddof=0)
    q1, q3 = _VAL.quantile([0.25, 0.75])
    bajo, alto = q1 - 1.5 * (q3 - q1), q3 + 1.5 * (q3 - q1)
    yield "`z` divide por el desvío con divisor n", _cerca(g["z"].std(ddof=0), 1)
    yield "`marcados_z`", g["marcados_z"] == (z.abs() > 3).sum()
    yield "`limite_inferior` y `limite_superior`", _cerca(g["limite_inferior"], bajo) and _cerca(g["limite_superior"], alto)
    yield "`marcados_ric`", g["marcados_ric"] == ((_VAL < bajo) | (_VAL > alto)).sum()


def _pruebas_10(g):
    yield "`conteo_region` cuenta las personas por región", (
        g["conteo_region"].to_dict() == personas["region"].value_counts().to_dict())
    yield ("`inicio_eje`" + (": escribí un número, sin comillas" if isinstance(g["inicio_eje"], str) else ""),
           _numero(g["inicio_eje"]) and _cerca(g["inicio_eje"], 0))
    orden = g["orden_nivel"]
    yield "`orden_nivel` no tiene huecos", not any(_hueco(o) for o in orden)
    yield "`orden_nivel` va de sin instrucción a superior completo", list(orden) == _ORDEN


def _pruebas_11(g):
    bins = g["bins_histograma"]
    yield "el histograma usa los intervalos del ejercicio 4", np.ndim(bins) == 1 and list(bins) == list(range(0, 151, 10))
    yield "`con_ingreso` no tiene el -9", len(g["con_ingreso"]) == len(_VAL)
    mediana = _OC[_ING != -9].groupby("nivel_educativo")["ingreso_ocupacion_principal"].median()
    yield "`mediana_por_nivel`", all(_cerca(g["mediana_por_nivel"].get(k), v) for k, v in mediana.items())


def _pruebas_12(g):
    yield "`serie_mediana` es la columna de la mediana", list(g["serie_mediana"]) == list(trimestres["ingreso_mediano"])
    yield "`serie_media` es la columna de la media", list(g["serie_media"]) == list(trimestres["ingreso_medio"])
    yield "`veces`: cuántas veces creció la mediana", _cerca(
        g["veces"], trimestres["ingreso_mediano"].iloc[-1] / trimestres["ingreso_mediano"].iloc[0])


def _pruebas_13(g):
    region = g["region_elegida"]
    regiones = sorted(personas["region"].unique())
    if region not in regiones:
        yield f"`region_elegida`: tiene que ser una de estas, escrita igual: {', '.join(regiones)}", False
        return
    x = _VAL[_OC.loc[_VAL.index, "region"] == region]
    q1, q3 = x.quantile([0.25, 0.75])
    ric = q3 - q1
    yield f"`casos_validos` en {region}", g["casos_validos"] == len(x)
    yield "`mediana_region`", _cerca(g["mediana_region"], x.median())
    yield "`ric_region`", _cerca(g["ric_region"], ric)
    yield "`atipicos_region`", g["atipicos_region"] == ((x < q1 - 1.5 * ric) | (x > q3 + 1.5 * ric)).sum()


# Qué ejercicios anteriores usa cada uno, para avisar de dónde viene un error.
_USA = {5: [2], 7: [2, 5], 8: [2, 5, 7], 9: [2, 5, 7], 11: [4, 10], 13: [11]}
_ULTIMA = {"error": None}


def _registrar(resultado=None):
    if resultado is not None and not resultado.info.raw_cell.lstrip().startswith("#@title Control del"):
        _ULTIMA["error"] = resultado.error_in_exec or resultado.error_before_exec


_ip = get_ipython()
_ip.events.callbacks["post_run_cell"] = [f for f in _ip.events.callbacks["post_run_cell"]
                                         if getattr(f, "__name__", "") != "_registrar"]
_ip.events.register("post_run_cell", _registrar)


def _sin_errores(numero, g):
    try:
        return all(bool(ok) for _, ok in globals()[f"_pruebas_{numero}"](g))
    except Exception:
        return False


def controlar(numero):
    g = globals()
    pruebas = globals()[f"_pruebas_{numero}"](g)
    lineas, bien = [], 0
    while True:
        try:
            texto, ok = next(pruebas)
        except StopIteration:
            break
        except KeyError as e:
            lineas.append(f"✘ Todavía no existe la variable {e}. Completá los huecos y ejecutá la celda del ejercicio {numero}.")
            break
        except Exception:
            lineas.append(f"✘ Lo que sigue no se puede revisar: la celda del ejercicio {numero} dejó un resultado de otro tipo.")
            break
        try:
            ok = bool(ok)
        except (TypeError, ValueError):
            ok = False
        lineas.append(("✔ " if ok else "✘ ") + texto)
        bien += ok
    if len(lineas) == 1 and lineas[0].startswith("✘ Todavía"):
        print(lineas[0][2:])
        return
    if _ULTIMA["error"] is not None:
        print(f"Ojo: la última celda que ejecutaste terminó con un error ({type(_ULTIMA['error']).__name__}). "
              "Corregila y ejecutala de nuevo, porque lo que sigue puede venir de una ejecución anterior.\n")
    for previo in _USA.get(numero, []):
        if not _sin_errores(previo, g):
            print(f"Ojo: este ejercicio usa resultados del ejercicio {previo}, que no está completo o tiene algo "
                  f"marcado con ✘. Corregí primero el ejercicio {previo}.\n")
    print("\n".join(lineas))
    print()
    print(f"Ejercicio {numero}: {bien} de {len(lineas)}." +
          (" Todo bien." if bien == len(lineas) else " Revisá lo marcado con ✘."))


print("Controles listos.")


## Los datos

Los datos están en tres tablas.

| Tabla | Filas | Una fila es |
|---|---|---|
| `personas` | 46.086 | una persona encuestada |
| `hogares` | 16.169 | un hogar |
| `trimestres` | 12 | un trimestre, de 2023 a 2025 |

Las columnas de `personas`:

| Columna | Qué es |
|---|---|
| `id_hogar` y `componente` | el número del hogar y el de la persona dentro del hogar |
| `region` | la región: Gran Buenos Aires, Noroeste, Noreste, Cuyo, Pampeana o Patagonia |
| `sexo` | `varón` o `mujer` |
| `edad` | años cumplidos |
| `nivel_educativo` | el máximo nivel alcanzado, de `sin instrucción` a `superior o universitario completo` |
| `condicion_actividad` | `ocupado`, `desocupado`, `inactivo`, `menor de 10 años` o `entrevista individual no realizada` |
| `categoria_ocupacional` | solo ocupados: `patrón`, `cuenta propia`, `obrero o empleado` o `trabajador familiar sin remuneración` |
| `horas_semanales` | solo ocupados: horas trabajadas en la semana en la ocupación principal |
| `ingreso_ocupacion_principal` | solo ocupados: ingreso mensual de la ocupación principal, en pesos |

En las personas que no están ocupadas, las tres últimas columnas están vacías, porque la pregunta
no les corresponde.

**Las horas y el ingreso conservan dos códigos del INDEC.** `999` en las horas quiere decir que la
persona no sabe o no respondió, y `-9` en el ingreso, que no respondió. No son cantidades, y el
ejercicio 2 trata sobre ellos.

La tabla `hogares` tiene, entre otras columnas, la `region` y la cantidad de `miembros` de cada
hogar. La tabla
`trimestres` resume doce encuestas, de 2023 a 2025: cuántas personas se encuestaron, cuántos
ocupados hubo, y la mediana y la media del ingreso de la ocupación principal, en pesos de cada
momento.

**Los números describen a las personas encuestadas.** El INDEC calcula sus cifras oficiales
multiplicando cada respuesta por la cantidad de personas que representa. Acá no se hace eso, así que
los resultados describen a la muestra y no a toda la población, que es lo que corresponde a la
unidad 4.


---
# 0. Primeros pasos

Antes de los ejercicios, lo mínimo para leer y escribir las celdas: cinco celdas cortas que ya
funcionan, una tabla con los errores más comunes y la forma de llenar un hueco. Ejecutá las celdas
en orden y mirá qué devuelve cada una.

### Una celda hace una cuenta

Una celda de código se ejecuta con `Shift + Enter`, y muestra abajo el resultado de su última
línea. Si esa línea pone un nombre con `=`, no muestra nada, y está bien. Con el signo `=` se le
pone **nombre** a un resultado, para usarlo después. El nombre se llama **variable**: `total` vale
12 hasta que se le asigne otra cosa.

Si un resultado aparece como `np.int64(3288)` o `np.float64(35.36)`, el número es el que está entre
paréntesis.


In [ ]:
total = 5 + 7
total * 2


### La tabla

Los datos están en una **tabla** llamada `personas`, una fila por persona encuestada. En `pandas`,
la herramienta de Python para tablas, una tabla se llama `DataFrame`. `personas.head()` muestra
las primeras cinco filas.


In [ ]:
personas.head()


### Una columna

Una columna se elige con su nombre entre corchetes y comillas: `personas["edad"]`. El nombre tiene
que estar escrito **exactamente** igual que en la tabla, sin mayúsculas ni acentos de más.

Sobre una columna se piden medidas con un punto y el nombre de la medida, seguido de paréntesis
vacíos. Las que más vas a usar son `.mean()` para la media, `.median()` para la mediana, `.min()` y
`.max()` para el mínimo y el máximo, y `.std(ddof=0)` para el desvío con divisor `n`, el del
apunte.


In [ ]:
print("Edad media:", personas["edad"].mean())
print("Edad mediana:", personas["edad"].median())
print("Edad mínima y máxima:", personas["edad"].min(), personas["edad"].max())


`print(...)` muestra en pantalla lo que va entre paréntesis. Sirve para mostrar varias cosas en una
misma celda, y para acompañar cada número con un texto entre comillas que diga qué es.
`round(numero, 2)` lo redondea a dos decimales.

### Contar categorías

`.value_counts()` cuenta cuántas veces aparece cada valor de una columna, ordenado de mayor a menor.
Es el primer paso de una serie de frecuencia.


In [ ]:
personas["region"].value_counts()


### Quedarse con algunas filas

Para quedarse con las filas que cumplen una condición se escribe la condición entre corchetes. Se
lee así: de `personas`, las filas donde la columna `sexo` es igual a `"mujer"`.

El **doble igual** `==` pregunta si dos cosas son iguales; el igual simple `=` pone un nombre.
También se puede preguntar con `!=`, distinto de, y con `<`, `>`, `<=` y `>=`. `len(...)` dice
cuántas filas tiene una tabla.


In [ ]:
varones = personas[personas["sexo"] == "varón"]
print("Varones encuestados:", len(varones))
print("Edad media de los varones:", round(varones["edad"].mean(), 2))


### Cuando algo sale mal

Los errores en rojo son normales y no rompen nada: se corrige la celda y se vuelve a ejecutar. Los
que más vas a ver:

| Error | Qué quiere decir |
|---|---|
| `NameError: name 'COMPLETAR' is not defined` o `KeyError: 'COMPLETAR'` | quedó un hueco sin completar |
| `AttributeError: 'Series' object has no attribute 'COMPLETAR'` | quedó sin completar un hueco como `.COMPLETAR()`: va el nombre de la medida |
| `KeyError: 'Edad'` | el nombre de la columna está mal escrito: acá es `edad`, en minúscula |
| `NameError: name 'horas' is not defined` | la variable se crea en una celda anterior que no se ejecutó: ejecutá las anteriores |

### Cómo se llena un hueco

Se borra solo la palabra `COMPLETAR`, y lo que la rodea queda igual:

- `"COMPLETAR"`, entre comillas: va un texto, entre esas mismas comillas. `personas["COMPLETAR"]`
  pasa a `personas["edad"]`.
- `.COMPLETAR()`: va solo el nombre de la medida, porque el punto y los paréntesis ya están.
  `columna.COMPLETAR()` pasa a `columna.mean()`.
- `COMPLETAR` sin comillas: va un número, el nombre de una variable, una cuenta corta, o `True` o
  `False`, sin agregar comillas.

En el código, los decimales van con punto y los miles no llevan separador: `0.75` y `13847`, no
`0,75` ni `13.847`. Python lee la coma como una separación entre dos números.

### Ahora vos

Calculá cuántas mujeres hay en la encuesta y cuál es su edad media. Es el ejemplo de los varones
con tres huecos: en dos cambia algo respecto del ejemplo, y el tercero se copia igual.


In [ ]:
mujeres = personas[personas["sexo"] == "COMPLETAR"]
cantidad_de_mujeres = len(COMPLETAR)
edad_media_mujeres = mujeres["COMPLETAR"].mean()

print("Mujeres encuestadas:", cantidad_de_mujeres)
print("Edad media de las mujeres:", round(edad_media_mujeres, 2))


In [ ]:
#@title Control del ejercicio 0 { run: "auto", display-mode: "form" }

if "controlar" in globals():
    controlar(0)
else:
    print("Primero ejecutá las tres celdas de Preparación, al principio del notebook.")


---
# 1. Qué es cada variable

*Apunte, secciones 1.2 y 1.4.*

La escala de una variable decide qué medidas y qué gráficos tienen sentido, así que es lo primero
que hay que decidir.

### Cómo se hace

Se miran los valores. `.unique()` muestra los valores distintos de una columna, y `.describe()`
resume una columna numérica. Con eso se contestan dos preguntas:

1. ¿Los valores son **categorías** o **números**? Si son categorías sin orden, la escala es
   nominal; si tienen un orden propio, es ordinal.
2. Si son números, ¿el cero quiere decir "nada"? Si lo quiere decir, la escala es de razón, y un
   valor puede ser el doble de otro. Si el cero es un punto elegido por convención, la escala es de
   intervalo.

Un ejemplo de cada una, con variables que no están en el ejercicio:

- `categoria_ocupacional`: categorías sin orden. **Nominal.**
- La satisfacción del relevamiento del apunte, `mala`, `regular` y `buena`: categorías con orden,
  pero no se puede medir cuánto más es `buena` que `regular`. **Ordinal.**
- Una temperatura en grados: 0 °C no quiere decir "sin temperatura", y 20 °C no es el doble de
  10 °C. **De intervalo.**
- `miembros`, en la tabla `hogares`: 0 sería ningún miembro, y 4 es el doble de 2. **De razón.**


In [ ]:
print(personas["nivel_educativo"].unique())
personas["edad"].describe()


### Ahora vos

Completá la escala de cada columna: `"nominal"`, `"ordinal"`, `"de intervalo"` o `"de razón"`. A la
izquierda de cada `:` está la columna, y a la derecha va su escala, entre las comillas, en lugar de
`COMPLETAR`.


In [ ]:
escala = {
    "region": "COMPLETAR",
    "sexo": "COMPLETAR",
    "nivel_educativo": "COMPLETAR",
    "edad": "COMPLETAR",
    "horas_semanales": "COMPLETAR",
    "ingreso_ocupacion_principal": "COMPLETAR",
}


In [ ]:
#@title Control del ejercicio 1 { run: "auto", display-mode: "form" }

if "controlar" in globals():
    controlar(1)
else:
    print("Primero ejecutá las tres celdas de Preparación, al principio del notebook.")


---
# 2. El denominador: vacíos, centinelas y ceros

*Apunte, sección 2.1.*

Las horas y el ingreso solo existen para las personas **ocupadas**, y las dos columnas traen los
códigos del INDEC. En las horas, `999` quiere decir que la persona no sabe o no respondió: es un
**centinela**, una ausencia disfrazada de número. El `0`, en cambio, es una persona ocupada que esa
semana no trabajó, por ejemplo por licencia. Es un dato, y se queda.

### Cómo se hace

El ejemplo trabaja con las horas:

1. Se queda con los ocupados, filtrando como en el ejercicio 0.
2. Cuenta los `999` y los ceros. `(columna == 999)` responde verdadero o falso fila por fila, y
   `.sum()` cuenta los verdaderos.
3. Convierte el centinela en faltante con `.replace(999, np.nan)`, donde `np.nan` es la celda vacía,
   y descarta los faltantes con `.dropna()`. Los ceros no se tocan.
4. Compara la media con el centinela adentro y la media correcta. `num(valor, 2)` muestra el
   número con coma decimal, punto de miles y dos decimales, como en el apunte.


In [ ]:
ocupados = personas[personas["condicion_actividad"] == "ocupado"]
horas_crudas = ocupados["horas_semanales"]

print("Ocupados:", num(len(ocupados), 0))
print("Cuántos 999 hay:", (horas_crudas == 999).sum())
print("Cuántos 0 hay:", (horas_crudas == 0).sum())

horas = horas_crudas.replace(999, np.nan).dropna()

print("Media con el 999 adentro:", num(horas_crudas.mean(), 2))
print("Media correcta:", num(horas.mean(), 2), "horas, sobre", num(len(horas), 0), "ocupados")


### Ahora vos

Lo mismo con el ingreso de la ocupación principal. Según el INDEC, ahí el centinela es `-9`, y el
`0` quiere decir "sin ingresos": la persona no cobró nada por esa ocupación en el mes de referencia.
Es un ingreso real que vale cero, no una falta de respuesta. La celda calcula
además una media equivocada a propósito, descartando los ceros, para ver cuánto cambia.


In [ ]:
ingreso = ocupados["COMPLETAR"]

cantidad_centinelas = (ingreso == COMPLETAR).sum()
cantidad_ceros = (ingreso == COMPLETAR).sum()

ingreso_valido = ingreso.replace(COMPLETAR, np.nan).dropna()

media_con_centinela = ingreso.mean()
media_sin_ceros = ingreso_valido[ingreso_valido != 0].mean()
media_correcta = ingreso_valido.COMPLETAR()

print("No respondieron:", num(cantidad_centinelas, 0), "  Declararon cero:", num(cantidad_ceros, 0))
print("Media contando el -9 como un ingreso:", num(media_con_centinela, 0))
print("Media descartando también los ceros: ", num(media_sin_ceros, 0))
print("Media correcta:", num(media_correcta, 0), "sobre", num(len(ingreso_valido), 0), "ocupados")


In [ ]:
#@title Control del ejercicio 2 { run: "auto", display-mode: "form" }

if "controlar" in globals():
    controlar(2)
else:
    print("Primero ejecutá las tres celdas de Preparación, al principio del notebook.")


**Pregunta.** ¿Cuál de las tres medias informarías, y con qué denominador? ¿Qué parte de los
ocupados queda afuera, y por qué hay que decirlo?


*Tu respuesta:*


---
# 3. La serie de frecuencia sin agrupar

*Apunte, sección 2.2.*

Una variable que toma pocos valores se resume contando cuántas veces aparece cada uno. La serie de
frecuencia tiene cuatro columnas: `fᵢ` cuenta los casos, `frᵢ` los pasa a proporción, y `Fᵢ` y
`Frᵢ` los van acumulando.

### Cómo se hace

El ejemplo arma la serie de la edad de los menores de 10 años, que toma los valores de 0 a 9:

- `.value_counts()` cuenta, y `.sort_index()` ordena las filas por el valor, de menor a mayor. Sin
  ese paso quedarían ordenadas por frecuencia, y las acumuladas no tendrían sentido.
- Dividir `fi` por `fi.sum()`, el total, da la frecuencia relativa.
- `.cumsum()` acumula: cada fila suma las anteriores.
- `pd.DataFrame({...})` junta las cuatro columnas en una tabla, y `display(...)` la muestra.


In [ ]:
chicos = personas[personas["edad"] < 10]

fi_edad = chicos["edad"].value_counts().sort_index()
fri_edad = fi_edad / fi_edad.sum()
Fi_edad = fi_edad.cumsum()
Fri_edad = Fi_edad / fi_edad.sum()

display(pd.DataFrame({"fᵢ": fi_edad, "frᵢ": fri_edad.round(4), "Fᵢ": Fi_edad, "Frᵢ": Fri_edad.round(4)}))


### Ahora vos

Armá la serie de la cantidad de **miembros del hogar**. Se calcula sobre la tabla `hogares`, no
sobre `personas`, para contar cada hogar una sola vez.

Después leé en la tabla dos cosas, sin calcular nada más, y escribilas como números:

- la **mediana**: la cantidad de miembros, en la columna `miembros` de la izquierda, de la primera
  fila donde `Frᵢ` llega a 0,5 o lo pasa;
- cuántos hogares tienen **cuatro miembros o menos**: el `Fᵢ` de la fila del 4, sin el punto de los
  miles.


In [ ]:
fi = hogares["COMPLETAR"].value_counts().COMPLETAR()
fri = fi / COMPLETAR
Fi = fi.COMPLETAR()
Fri = Fi / fi.sum()

display(pd.DataFrame({"fᵢ": fi, "frᵢ": fri.round(4), "Fᵢ": Fi, "Frᵢ": Fri.round(4)}))

mediana_leida = COMPLETAR
hogares_de_hasta_4 = COMPLETAR


In [ ]:
#@title Control del ejercicio 3 { run: "auto", display-mode: "form" }

if "controlar" in globals():
    controlar(3)
else:
    print("Primero ejecutá las tres celdas de Preparación, al principio del notebook.")


---
# 4. La serie con intervalos

*Apunte, secciones 2.3 y 2.4.*

Cuando la variable toma muchos valores, se agrupa en **intervalos de clase**. Tres decisiones:
cuántos intervalos, de qué amplitud y dónde empieza el primero. Y una convención: cerrado a la
izquierda y abierto a la derecha, `[20, 30)`, que incluye al 20 y no al 30.

### Cómo se hace

El ejemplo agrupa la edad de los ocupados en intervalos de 10 años, desde 10 hasta 100:

- La regla de Sturges, `1 + 3,322 × log₁₀(n)`, sugiere cuántos intervalos usar. En Python el
  logaritmo en base 10 es `np.log10(...)`.
- `list(range(10, 101, 10))` arma la lista de bordes: 10, 20, …, 100. El segundo número es hasta
  dónde llegar, y se pone uno más que el último borde porque `range` no lo incluye.
- `pd.cut(columna, bins=bordes, right=False)` pone a cada valor en su intervalo. El `right=False`
  es la convención de la asignatura: **sin él, `pandas` cierra los intervalos a la derecha**.
- La marca de clase es el punto medio de cada intervalo, el borde izquierdo más 5. La media
  agrupada multiplica cada marca por su frecuencia, suma, y divide por `n`.
- `bordes_edad[:-1]` son todos los bordes menos el último, que no abre ningún intervalo, y
  `.to_numpy()` pasa las frecuencias a una lista de números para multiplicarlas por las marcas.
- Para la edad, Sturges sugiere 15 intervalos y el ejemplo usa 9, de 10 años cada uno, porque se
  leen mejor. La regla es una sugerencia: el criterio final es que la tabla se lea (sección 2.3 del
  apunte).


In [ ]:
edad_ocupados = ocupados["edad"]
n_edad = len(edad_ocupados)
print("Sturges sugiere", num(1 + 3.322 * np.log10(n_edad), 2), "intervalos")

bordes_edad = list(range(10, 101, 10))
clases_edad = pd.cut(edad_ocupados, bins=bordes_edad, right=False)
fi_edad_ocupados = clases_edad.value_counts().sort_index()

marcas_edad = np.array(bordes_edad[:-1]) + 5
media_agrupada_edad = (marcas_edad * fi_edad_ocupados.to_numpy()).sum() / n_edad

display(pd.DataFrame({"marca de clase": marcas_edad, "fᵢ": fi_edad_ocupados.to_numpy()},
                     index=fi_edad_ocupados.index.astype(str)))
print("Media agrupada:", num(media_agrupada_edad, 2), "  Media real:", num(edad_ocupados.mean(), 2))


### Ahora vos

Agrupá las `horas` del ejercicio 2, que ya no tienen el `999`, en intervalos de 10 horas que
empiecen en 0. El valor más alto es 140: elegí el último borde **más chico** que deje a todos los
datos adentro, así ningún dato queda afuera y no sobra ningún intervalo. Ojo, que con intervalos
abiertos a la derecha el 140 no entra en `[130, 140)`.


In [ ]:
n = len(horas)
k_sturges = 1 + 3.322 * np.log10(COMPLETAR)
print("Sturges sugiere", num(k_sturges, 2), "intervalos")

ultimo_borde = COMPLETAR
bordes = list(range(0, ultimo_borde + 1, 10))
clases_horas = pd.cut(horas, bins=bordes, right=COMPLETAR)
fi_horas = clases_horas.value_counts().sort_index()

marcas = np.array(bordes[:-1]) + 5
media_agrupada_horas = (marcas * fi_horas.to_numpy()).sum() / COMPLETAR

display(pd.DataFrame({"marca de clase": marcas, "fᵢ": fi_horas.to_numpy()},
                     index=fi_horas.index.astype(str)))
print("Media agrupada:", num(media_agrupada_horas, 2), "  Media real:", num(horas.mean(), 2))


In [ ]:
#@title Control del ejercicio 4 { run: "auto", display-mode: "form" }

if "controlar" in globals():
    controlar(4)
else:
    print("Primero ejecutá las tres celdas de Preparación, al principio del notebook.")


**Pregunta.** En el ejemplo de la edad, la media agrupada y la real casi coinciden. En las horas, la
agrupada queda casi dos horas arriba. La celda de abajo cuenta cuántas personas declararon
exactamente 40 horas. ¿En qué parte del intervalo `[40, 50)` caen, y por qué la marca de clase 45
las representa mal?


In [ ]:
(horas == 40).sum()


*Tu respuesta:*


---
# 5. Las tres medidas de posición

*Apunte, secciones 3.1 a 3.4.*

### Cómo se hace

Sobre las horas:

- `.mean()` y `.median()` dan la media y la mediana.
- `.mode()` da el modo. Devuelve una lista, porque puede haber más de un modo, y `.iloc[0]` toma el
  primero.
- `(horas < media_horas)` responde verdadero o falso para cada persona, y `.mean()` sobre eso da la
  proporción de verdaderos: qué parte trabaja menos que la media.


In [ ]:
media_horas = horas.mean()
mediana_horas = horas.median()
modo_horas = horas.mode().iloc[0]

print("Media:", num(media_horas, 2), "  Mediana:", num(mediana_horas, 0), "  Modo:", num(modo_horas, 0))
print("Trabaja menos que la media:", num(100 * (horas < media_horas).mean(), 1) + "%")


### Ahora vos

Las mismas cuatro cuentas sobre `ingreso_valido`, el ingreso sin el `-9` del ejercicio 2. En la
última línea, escribí sin comillas el nombre de la medida que informarías como ingreso típico:
`media`, `mediana` o `modo`. La regla está en la sección 3.4 del apunte.


In [ ]:
media = ingreso_valido.COMPLETAR()
mediana = ingreso_valido.COMPLETAR()
modo = ingreso_valido.COMPLETAR().iloc[0]
proporcion_bajo_la_media = (ingreso_valido < COMPLETAR).mean()
ingreso_tipico = COMPLETAR

print("Media:", num(media, 0), "  Mediana:", num(mediana, 0), "  Modo:", num(modo, 0))
print("Gana menos que la media:", num(100 * proporcion_bajo_la_media, 1) + "%")


In [ ]:
#@title Control del ejercicio 5 { run: "auto", display-mode: "form" }

if "controlar" in globals():
    controlar(5)
else:
    print("Primero ejecutá las tres celdas de Preparación, al principio del notebook.")


**Pregunta.** ¿Por qué esa medida, y no las otras dos, es el ingreso típico? En el relevamiento
del apunte el ingreso no tenía modo, porque los 35 valores eran distintos. Acá el modo lo comparten
más de mil personas: ¿qué tienen en común los valores más repetidos? La celda de abajo muestra los
diez más repetidos.


In [ ]:
ingreso_valido.value_counts().head(10)


*Tu respuesta:*


---
# 6. Los cuartiles, a mano y con pandas (opcional)

*Apunte, sección 3.5.*

La fórmula clásica ubica el cuartil `k` en la posición `k(n + 1)/4` de la serie ordenada. `pandas` y
`numpy` usan otro método, y con pocos datos los resultados se separan.

### Cómo se hace

Con siete valores, `n + 1 = 8`, así que las posiciones son `1 × 8/4 = 2`, `2 × 8/4 = 4` y
`3 × 8/4 = 6`. La herramienta `en_posicion(valores, posicion)` devuelve el valor que ocupa esa
posición, contando desde 1 como el apunte. `np.percentile(valores, [25, 50, 75])` da los cuartiles
de `numpy`.


In [ ]:
siete = [3, 5, 5, 6, 8, 9, 15]

print("Fórmula:", en_posicion(siete, 2), en_posicion(siete, 4), en_posicion(siete, 6))
print("numpy:  ", np.percentile(siete, [25, 50, 75]))


### Ahora vos

La celda toma los ingresos de los primeros once ocupados de la Patagonia que declararon su
ingreso, y los ordena. Calculá las tres posiciones con `n = 11` y escribilas en los huecos como
números enteros, por ejemplo `3`.


In [ ]:
patagonia = ocupados[(ocupados["region"] == "Patagonia") & (ocupados["ingreso_ocupacion_principal"] != -9)]
once = np.sort(patagonia["ingreso_ocupacion_principal"].head(11).to_numpy())
print("Los once valores:", [num(v, 0) for v in once])

q1_formula = en_posicion(once, COMPLETAR)
q2_formula = en_posicion(once, COMPLETAR)
q3_formula = en_posicion(once, COMPLETAR)
q1_numpy, q2_numpy, q3_numpy = np.percentile(once, [25, 50, 75])

display(pd.DataFrame({"fórmula k(n+1)/4": [num(v, 0) for v in (q1_formula, q2_formula, q3_formula)],
                      "numpy y pandas": [num(v, 0) for v in (q1_numpy, q2_numpy, q3_numpy)]},
                     index=["Q₁", "Q₂", "Q₃"]))


In [ ]:
#@title Control del ejercicio 6 { run: "auto", display-mode: "form" }

if "controlar" in globals():
    controlar(6)
else:
    print("Primero ejecutá las tres celdas de Preparación, al principio del notebook.")


---
# 7. Las medidas de variabilidad

*Apunte, secciones 4.1 a 4.5.*

### Cómo se hace

Sobre la edad de los ocupados:

- El rango es el máximo menos el mínimo.
- `.std(ddof=0)` da el desvío con divisor `n`, el de las unidades 2 y 4. `.std()` sin nada divide
  por `n − 1`: es lo que hace `pandas` si no se le pide otra cosa.
- `.quantile([0.25, 0.75])` da los dos cuartiles juntos. Con `q1_edad, q3_edad = ...` cada uno queda
  con su nombre.
- El `RIC` es `Q₃ − Q₁`, y el coeficiente de variabilidad es el desvío sobre la media.


In [ ]:
rango_edad = edad_ocupados.max() - edad_ocupados.min()
desvio_edad = edad_ocupados.std(ddof=0)
q1_edad, q3_edad = edad_ocupados.quantile([0.25, 0.75])
ric_edad = q3_edad - q1_edad
cv_edad = desvio_edad / edad_ocupados.mean()

print("Rango:", num(rango_edad, 0), "  Desvío:", num(desvio_edad, 2), "  RIC:", num(ric_edad, 0))
print("Desvío con n − 1:", num(edad_ocupados.std(), 2))
print("CV:", num(100 * cv_edad, 2) + "%")


### Ahora vos

Lo mismo sobre `ingreso_valido`. Para el coeficiente de variabilidad, usá la `media` del ejercicio
5. En la última línea, escribí sin comillas la medida que describe mejor la dispersión del ingreso:
`rango`, `desvio_n` o `ric`. La regla está en la sección 4.4 del apunte.


In [ ]:
rango = ingreso_valido.max() - ingreso_valido.COMPLETAR()
desvio_n = ingreso_valido.std(ddof=COMPLETAR)
q1, q3 = ingreso_valido.quantile([0.25, COMPLETAR])
ric = COMPLETAR
cv_ingreso = desvio_n / COMPLETAR
dispersion_tipica = COMPLETAR

print("Rango:", num(rango, 0), "  Desvío:", num(desvio_n, 0), "  RIC:", num(ric, 0))
print("Desvío con n − 1:", num(ingreso_valido.std(), 0))
print("CV del ingreso:", num(100 * cv_ingreso, 2) + "%")
print("CV de las horas:", num(100 * horas.std(ddof=0) / horas.mean(), 2) + "%")


In [ ]:
#@title Control del ejercicio 7 { run: "auto", display-mode: "form" }

if "controlar" in globals():
    controlar(7)
else:
    print("Primero ejecutá las tres celdas de Preparación, al principio del notebook.")


**Pregunta.** En el relevamiento del apunte, con 35 ingresos, pasar de `n` a `n − 1` cambiaba el
desvío un 1,46%. ¿Cuánto lo cambia acá, y por qué? ¿Por qué elegiste esa medida para la
dispersión? ¿Qué varía relativamente más, el ingreso o las horas?


*Tu respuesta:*


---
# 8. La asimetría y la kurtosis (opcional)

*Apunte, secciones 5.1 a 5.3.*

### Cómo se hace

Sobre la edad de los ocupados:

- `stats.skew(...)` da la asimetría `α₃`.
- `stats.kurtosis(..., fisher=False)` da la kurtosis `α₄`, la que vale 3 en la normal. Sin el
  `fisher=False` devuelve el exceso, que es `α₄ − 3`: es la advertencia de la sección 5.3.
- Los dos coeficientes de Pearson son `Ap₁ = (media − modo) / desvío` y
  `Ap₂ = 3 × (media − mediana) / desvío`.


In [ ]:
media_edad = edad_ocupados.mean()
mediana_edad = edad_ocupados.median()
modo_edad = edad_ocupados.mode().iloc[0]

print("α₃:", num(stats.skew(edad_ocupados), 4))
print("α₄:", num(stats.kurtosis(edad_ocupados, fisher=False), 4))
print("Ap₁:", num((media_edad - modo_edad) / desvio_edad, 4))
print("Ap₂:", num(3 * (media_edad - mediana_edad) / desvio_edad, 4))


### Ahora vos

Lo mismo sobre `ingreso_valido`, con la `media`, la `mediana`, el `modo` y el `desvio_n` de los
ejercicios 5 y 7. En cada coeficiente de Pearson falta la medida de posición que se compara con la
media.


In [ ]:
a3 = stats.skew(ingreso_valido)
a4 = stats.kurtosis(ingreso_valido, fisher=COMPLETAR)
ap1 = (media - COMPLETAR) / desvio_n
ap2 = 3 * (media - COMPLETAR) / desvio_n

print("α₃:", num(a3, 4), "  α₄:", num(a4, 4))
print("Ap₁:", num(ap1, 4), "  Ap₂:", num(ap2, 4))


In [ ]:
#@title Control del ejercicio 8 { run: "auto", display-mode: "form" }

if "controlar" in globals():
    controlar(8)
else:
    print("Primero ejecutá las tres celdas de Preparación, al principio del notebook.")


**Pregunta.** En la tabla de la sección 5.2 del apunte, con las personas por hogar del relevamiento,
los tres coeficientes de asimetría coinciden en el signo. ¿Coinciden acá? Si no, ¿qué medida de
posición produce la diferencia, y por qué no es confiable en esta variable?


*Tu respuesta:*


---
# 9. Los dos criterios de valor extremo

*Apunte, sección 6.*

### Cómo se hace

Sobre las horas:

- El **z-score** de cada valor es `(valor − media) / desvío`. Se calcula para toda la columna de una
  vez. `.abs()` saca el signo, y `(z.abs() > 3).sum()` cuenta cuántos pasan de 3.
- La regla del **`1,5 × RIC`** arma dos límites, y marca lo que cae afuera. El símbolo `|` quiere
  decir "o": por debajo del límite inferior **o** por encima del superior.


In [ ]:
z_horas = (horas - media_horas) / horas.std(ddof=0)
print("z-score marca:", (z_horas.abs() > 3).sum(), "valores")

q1_h, q3_h = horas.quantile([0.25, 0.75])
ric_h = q3_h - q1_h
inferior_h = q1_h - 1.5 * ric_h
superior_h = q3_h + 1.5 * ric_h
print("Límites del RIC:", num(inferior_h, 0), "y", num(superior_h, 0))
print("La regla del RIC marca:", ((horas < inferior_h) | (horas > superior_h)).sum(), "valores")


### Ahora vos

Los dos criterios sobre `ingreso_valido`. Usá la `media`, el `desvio_n`, `q1`, `q3` y `ric` de los
ejercicios anteriores.


In [ ]:
z = (ingreso_valido - media) / COMPLETAR
marcados_z = (z.abs() > COMPLETAR).sum()

limite_inferior = q1 - 1.5 * ric
limite_superior = COMPLETAR + 1.5 * ric
marcados_ric = ((ingreso_valido < limite_inferior) | (ingreso_valido > COMPLETAR)).sum()

print("z-score marca:", marcados_z, "valores")
print("Límites del RIC:", num(limite_inferior, 0), "y", num(limite_superior, 0))
print("La regla del RIC marca:", marcados_ric, "valores")


In [ ]:
#@title Control del ejercicio 9 { run: "auto", display-mode: "form" }

if "controlar" in globals():
    controlar(9)
else:
    print("Primero ejecutá las tres celdas de Preparación, al principio del notebook.")


**Pregunta.** ¿Por qué los dos criterios marcan cantidades tan distintas? En el relevamiento, el `0`
del hogar 38 quedaba marcado por el `RIC`. ¿Acá los ceros quedan marcados? ¿Qué harías con los
valores que marca cada criterio?


*Tu respuesta:*


---
# 10. El gráfico de barras

*Apunte, secciones 7.1 a 7.3.*

### Cómo se arma un gráfico

Todos los gráficos de este notebook siguen el mismo molde de cuatro pasos:

1. `fig, ax = plt.subplots()` crea un gráfico vacío. `ax` es el lugar donde se dibuja.
2. Una instrucción dibuja: `ax.bar(...)` barras, `ax.hist(...)` un histograma, `ax.plot(...)` una
   línea.
3. Otras instrucciones lo terminan: `ax.set_title("...")` pone el título, `ax.set_ylabel("...")` el
   nombre del eje vertical, y `ax.set_ylim(desde, hasta)` dice dónde empieza y termina ese eje.
4. `plt.show()` lo muestra.

Las celdas agregan otras opciones, como `figsize=(10, 4.5)` para el tamaño, `color=TEAL` para el
color o `ax.tick_params(axis="x", rotation=30)` para inclinar los nombres del eje. Solo cambian el
aspecto, y no hace falta tocarlas.

### Cómo se hace

El ejemplo dibuja la categoría ocupacional. Primero se cuenta con `.value_counts()`, y después
`ax.bar(categorías, alturas)` dibuja una barra por categoría. `conteo.index` son los nombres de las
categorías y `conteo.to_numpy()` son las cantidades. El eje vertical empieza en 0, como pide la
sección 7.2.


In [ ]:
conteo_categoria = ocupados["categoria_ocupacional"].value_counts()

fig, ax = plt.subplots(figsize=(10, 4.5))
ax.bar(conteo_categoria.index, conteo_categoria.to_numpy(), color=TEAL, width=0.6)
ax.set_title("Ocupados por categoría ocupacional")
ax.set_ylabel("ocupados")
ax.set_ylim(0, conteo_categoria.max() * 1.1)
plt.show()


### Ahora vos

Dos gráficos de barras, uno al lado del otro. `plt.subplots(1, 2)` crea dos lugares para dibujar,
`ax1` y `ax2`.

- A la izquierda, la **región**, que es nominal: va ordenada por frecuencia, que es como la deja
  `.value_counts()`.
- A la derecha, el **nivel educativo**, que es ordinal: va en su orden propio, del más bajo al más
  alto. `.reindex(orden_nivel)` reordena las filas según la lista. Completá las tres categorías que
  faltan. Copialas de la lista que mostró `.unique()` en el ejemplo del ejercicio 1, con las mismas
  minúsculas, tildes y espacios: si una está mal escrita no hay error, pero su barra no aparece.
- En `inicio_eje` va el valor donde empieza el eje vertical de los dos gráficos.


In [ ]:
conteo_region = personas["COMPLETAR"].value_counts()

orden_nivel = ["sin instrucción", "primario incompleto", "COMPLETAR", "secundario incompleto",
               "COMPLETAR", "COMPLETAR", "superior o universitario completo"]
conteo_nivel = personas["nivel_educativo"].value_counts().reindex(orden_nivel)

inicio_eje = COMPLETAR

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
ax1.bar(conteo_region.index, conteo_region.to_numpy(), color=TEAL, width=0.6)
ax1.set_title("Personas por región")
ax1.set_ylim(inicio_eje, conteo_region.max() * 1.1)
ax1.tick_params(axis="x", rotation=30)

ax2.bar(conteo_nivel.index, conteo_nivel.to_numpy(), color=CORAL, width=0.6)
ax2.set_title("Personas por nivel educativo")
ax2.set_ylim(inicio_eje, conteo_nivel.max() * 1.1)
ax2.tick_params(axis="x", rotation=35)
plt.setp(ax2.get_xticklabels(), ha="right")
plt.tight_layout()
plt.show()


In [ ]:
#@title Control del ejercicio 10 { run: "auto", display-mode: "form" }

if "controlar" in globals():
    controlar(10)
else:
    print("Primero ejecutá las tres celdas de Preparación, al principio del notebook.")


**Pregunta.** En la base del INDEC, `sin instrucción` tiene el código 7 y `primario incompleto` el
código 1. ¿Qué habría pasado si el gráfico se hubiera ordenado por el código? Y de `region`,
`nivel_educativo` y `categoria_ocupacional`, ¿para cuáles se podría usar un gráfico sectorial, para
cuál no, y por qué conviene igual el de barras?


*Tu respuesta:*


---
# 11. El histograma y el boxplot

*Apunte, secciones 7.4 y 7.6.*

### Cómo se hace

El histograma es `ax.hist(columna, bins=bordes)`: le das los datos sin contar y los bordes de los
intervalos, y cuenta solo. El ejemplo usa los `bordes_edad` del ejercicio 4.

El boxplot que compara grupos necesita los valores separados por grupo. La herramienta
`por_grupo(tabla, columna, grupo, orden)` los separa, y `caja(ax, grupos, nombres)` dibuja una caja
por grupo. El ejemplo compara las horas de varones y mujeres.

Para tener el número además del dibujo, `.groupby("sexo")["horas_semanales"].median()` calcula la
mediana de las horas para cada sexo: agrupa las filas por la columna `sexo` y hace la cuenta en cada
grupo.


In [ ]:
ocupados_con_horas = ocupados[ocupados["horas_semanales"] != 999]
grupos_sexo = por_grupo(ocupados_con_horas, "horas_semanales", "sexo", ["varón", "mujer"])

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(11, 7))
ax1.hist(edad_ocupados, bins=bordes_edad, color=TEAL, edgecolor=CREMA)
ax1.set_title("Edad de los ocupados")
ax1.set_ylabel("ocupados")
caja(ax2, grupos_sexo, ["varón", "mujer"])
ax2.set_title("Horas semanales, por sexo")
plt.tight_layout()
plt.show()

ocupados_con_horas.groupby("sexo")["horas_semanales"].median()


### Ahora vos

Arriba, el histograma de las `horas`, con los `bordes` del ejercicio 4. Abajo, un boxplot del
ingreso por nivel educativo, en el orden del ejercicio 10.

El eje del boxplot llega hasta 4.000.000, para que se vean las cajas. Hay ingresos más altos que
quedan fuera del dibujo, y el título dice cuántos.


In [ ]:
bins_histograma = COMPLETAR
con_ingreso = ocupados[ocupados["ingreso_ocupacion_principal"] != COMPLETAR]
grupos = por_grupo(con_ingreso, "ingreso_ocupacion_principal", "nivel_educativo", orden_nivel)
mediana_por_nivel = con_ingreso.groupby("COMPLETAR")["ingreso_ocupacion_principal"].median()

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(12, 9), gridspec_kw={"height_ratios": [1, 1.4]})
ax1.hist(horas, bins=bins_histograma, color=TEAL, edgecolor=CREMA)
ax1.set_title("Horas semanales de la ocupación principal")
ax1.set_ylabel("ocupados")
caja(ax2, grupos, orden_nivel)
ax2.set_xlim(0, 4_000_000)
ax2.xaxis.set_major_formatter(lambda valor, _: num(valor, 0))
fuera = (con_ingreso["ingreso_ocupacion_principal"] > 4_000_000).sum()
ax2.set_title(f"Ingreso por nivel educativo ({fuera} valores por encima de 4.000.000 quedan fuera del eje)")
plt.tight_layout()
plt.show()

mediana_por_nivel.reindex(orden_nivel)


In [ ]:
#@title Control del ejercicio 11 { run: "auto", display-mode: "form" }

if "controlar" in globals():
    controlar(11)
else:
    print("Primero ejecutá las tres celdas de Preparación, al principio del notebook.")


**Pregunta.** ¿Qué muestra el histograma de las horas que un boxplot no mostraría? ¿Cómo cambia la
mediana del ingreso con el nivel educativo? ¿En qué grupo desconfiarías de la comparación? Para
contestar la última, la celda de abajo cuenta los casos de cada nivel.


In [ ]:
con_ingreso["nivel_educativo"].value_counts().reindex(orden_nivel)


*Tu respuesta:*


---
# 12. El gráfico de líneas (opcional)

*Apunte, sección 7.5.*

### Cómo se hace

La tabla `trimestres` tiene una fila por trimestre, de 2023 a 2025. `ax.plot(x, y)` une los puntos
con una línea, y `marker="o"` marca cada punto. El ejemplo dibuja cuántas personas encuestó la EPH
en cada trimestre. `label="..."` le pone nombre a la línea, y `ax.legend()` muestra esos nombres.
`ax.set_ylim(0, None)` empieza el eje en cero y deja que `matplotlib` elija dónde termina.


In [ ]:
display(trimestres)

fig, ax = plt.subplots(figsize=(12, 4.5))
ax.plot(trimestres["periodo"], trimestres["personas_encuestadas"], marker="o", color=TINTA,
        label="personas encuestadas")
ax.set_ylim(0, None)
ax.set_title("Personas encuestadas por la EPH")
ax.legend()
ax.tick_params(axis="x", rotation=30)
plt.show()


### Ahora vos

Dos líneas en el mismo gráfico: la mediana y la media del ingreso. Mirá los nombres de las columnas
en la tabla de arriba. Después calculá cuántas veces creció la mediana: el último valor dividido por
el primero. `.iloc[-1]` toma el último y `.iloc[0]` el primero.


In [ ]:
serie_mediana = trimestres["COMPLETAR"]
serie_media = trimestres["COMPLETAR"]
veces = serie_mediana.iloc[-1] / serie_mediana.iloc[COMPLETAR]

fig, ax = plt.subplots(figsize=(12, 4.5))
ax.plot(trimestres["periodo"], serie_mediana, marker="o", color=CORAL, label="mediana")
ax.plot(trimestres["periodo"], serie_media, marker="o", color=TEAL, label="media")
ax.set_ylim(0, None)
ax.yaxis.set_major_formatter(lambda valor, _: num(valor, 0))
ax.set_title("Ingreso de la ocupación principal, en pesos corrientes")
ax.legend()
ax.tick_params(axis="x", rotation=30)
plt.show()

print("La mediana se multiplicó por", num(veces, 2))


In [ ]:
#@title Control del ejercicio 12 { run: "auto", display-mode: "form" }

if "controlar" in globals():
    controlar(12)
else:
    print("Primero ejecutá las tres celdas de Preparación, al principio del notebook.")


**Pregunta.** ¿Por qué acá corresponde un gráfico de líneas, y en el ejercicio 10 no? ¿Por qué la
media queda siempre arriba de la mediana? ¿Podés decir, con este gráfico, que los ocupados ganan
diez veces más que en 2023?


*Tu respuesta:*


---
# 13. El análisis descriptivo completo

*Apunte, secciones 8 y 11.*

El cierre junta todo. La celda arma la tabla de la sección 8 del apunte para el ingreso de los
ocupados de una región, con cuentas que ya hiciste en los ejercicios anteriores. Completá las tres
que faltan.

La celda usa `con_ingreso`, del ejercicio 11, y viene con la Patagonia. Cuando termines, probá con
otra región cambiando el nombre entre comillas, escrito exactamente así: `Gran Buenos Aires`,
`Noroeste`, `Noreste`, `Cuyo`, `Pampeana` o `Patagonia`.


In [ ]:
region_elegida = "Patagonia"

ocupados_region = ocupados[ocupados["region"] == region_elegida]
x = con_ingreso.loc[con_ingreso["region"] == region_elegida, "ingreso_ocupacion_principal"]

casos_validos = len(COMPLETAR)
mediana_region = x.COMPLETAR()
q1_region, q3_region = x.quantile([0.25, 0.75])
ric_region = COMPLETAR
atipicos_region = ((x < q1_region - 1.5 * ric_region) | (x > q3_region + 1.5 * ric_region)).sum()

display(pd.DataFrame({"valor": {
    "Casos válidos": f"{num(casos_validos, 0)} de {num(len(ocupados_region), 0)}",
    "Mínimo y máximo": f"{num(x.min(), 0)} y {num(x.max(), 0)}",
    "Media": num(x.mean(), 0),
    "Mediana": num(mediana_region, 0),
    "Modo": num(x.mode().iloc[0], 0),
    "Q₁ y Q₃": f"{num(q1_region, 0)} y {num(q3_region, 0)}",
    "Desvío standard (n)": num(x.std(ddof=0), 0),
    "RIC": num(ric_region, 0),
    "Coeficiente de variabilidad": num(100 * x.std(ddof=0) / x.mean(), 2) + "%",
    "Asimetría α₃": num(stats.skew(x), 4),
    "Kurtosis α₄": num(stats.kurtosis(x, fisher=False), 4),
    "Atípicos por 1,5 × RIC": atipicos_region,
}}))


In [ ]:
#@title Control del ejercicio 13 { run: "auto", display-mode: "form" }

if "controlar" in globals():
    controlar(13)
else:
    print("Primero ejecutá las tres celdas de Preparación, al principio del notebook.")


**El párrafo.** Escribilo con los números de la tabla, siguiendo el modelo de la sección 8 del
apunte. Tiene que tener cuatro cosas: el centro con dos medidas, la dispersión con dos medidas, los
valores extremos, y cuántos casos tienen el dato. Y tiene que hablar de los ocupados
**encuestados**: decir algo de todos los ocupados de la región es inferencia, y eso empieza en la
unidad 5. Antes de darlo por terminado, revisalo con la lista de verificación de la sección 11 del
apunte.


*Tu párrafo:*


---
# Cierre

Trece ejercicios sobre la unidad 4, tres de ellos opcionales. Los números y una respuesta posible
para cada pregunta están en la sección 7 de la consigna. Leela después de contestar, no antes.

> **Consigna, guía y soluciones:** `E8102_Ejercitacion_Bloque_2.pdf`
>
> **Apunte de la unidad 4:** `E8102_Apunte_U4_Analisis_de_Datos.pdf`
